In [1]:
import pickle
import sys
import CRPS.CRPS as pscore
import numpy as np
from pathlib import Path

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1"


import multiprocessing as mp
mp.set_start_method('spawn')

sys.path.insert(0, '../LSTM_next_activity_duration/notebooks/evaluation/')
sys.path.insert(0, '../../../../Evaluation')

import conduct_evaluation
import normal_evaluation.normal_evaluation
from prefix_duration_predictor import PrefixDurationPredictor, NOTEBOOK_DIR
from normal_evaluation.lstm_evaluation import SampleOutcomes_LSTM


get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [2]:
with open('../../../transformed_event_logs/BPIC_2017_all_test.pickle', 'rb') as f:
    test_data = pickle.load(f)


n_processes = 32
batch_size = 10
N = 1000

In [3]:

event_log_properties = {
    'case_name' : 'case:concept:name',
    'concept_name' : 'concept:name',
    'timestamp_name' : 'time:timestamp_start',
    'time_since_case_start_column' : '',
    'time_since_last_event_column' : '',
    'day_in_week_column' : 'day_in_week',
    'seconds_in_day_column' : 'seconds_in_day',
    'min_suffix_size' : 1,
    'train_validation_size' : 0.15,
    'test_validation_size' : 0.0,
    'window_size' : 'auto',
    'categorical_columns' : ['concept:name', 'org:resource_start'],
    'continuous_columns' : ['seconds_in_day', 'day_in_week', 'duration_seconds'],
    'continuous_positive_columns' : []
}

#NOTEBOOK_DIR = Path(__file__).resolve().parent
LSTM_ROOT = (NOTEBOOK_DIR / "../..").resolve()
LOADER_DIR = (NOTEBOOK_DIR / "../../../../load/event_log_loader").resolve()
ENCODED_DIR = (NOTEBOOK_DIR / "../../../../load/encoded_data").resolve()
TRANSFORMED_LOG_DIR = (NOTEBOOK_DIR / "../../../../../transformed_event_logs").resolve()
MODEL_DIR = (NOTEBOOK_DIR / "../training_variational_dropout/BPIC17").resolve()

TRAIN_DATA_PATH = (ENCODED_DIR / "BPIC_2017_all_1_train.pkl").resolve()

selected_cat_attributes = ['concept:name', 'org:resource_start']
selected_num_attributes = ['seconds_in_day', 'day_in_week']

lstm_predictor = PrefixDurationPredictor(
        train_loader_path = TRAIN_DATA_PATH,
        model_dir = MODEL_DIR,
        model_path = None,
        event_log_properties= event_log_properties,
        selected_cat_attributes = selected_cat_attributes,
        selected_num_attributes = selected_num_attributes,
        device = 'cpu'
)

Embeddings:  ModuleList(
  (0): Embedding(43, 16)
  (1): Embedding(149, 16)
)
Total embedding feature size:  32
Input feature size:  34
Cells hidden size:  128
Number of LSTM layer:  2
Dropout rate:  0.1




In [4]:
evaluator_A = conduct_evaluation.ConductEvaluation(lstm_predictor, SampleOutcomes_LSTM, {
                                                    },
                                    test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

  0%|                                                                                                                    | 0/6068 [00:00<?, ?it/s]

  1%|▉                                                                                                          | 50/6068 [00:01<02:14, 44.63it/s]

  2%|█▋                                                                                                        | 100/6068 [00:01<01:12, 82.30it/s]

  2%|██▌                                                                                                       | 150/6068 [00:01<00:59, 99.99it/s]

  3%|███▍                                                                                                     | 200/6068 [00:01<00:47, 124.58it/s]

...

 92%|█████████████████████████████████████████████████████████████████████████████████████▌       | 5581/6068 [00:17<00:01, 341.28it/s]

 94%|███████████████████████████████████████████████████████████████████████████████████████▋     | 5721/6068 [00:17<00:00, 468.25it/s]

 95%|████████████████████████████████████████████████████████████████████████████████████████▍    | 5771/6068 [00:18<00:01, 246.68it/s]

 99%|███████████████████████████████████████████████████████████████████████████████████████████▋ | 5981/6068 [00:18<00:00, 430.15it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████████| 6068/6068 [00:18<00:00, 324.33it/s]

In [5]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-7.002084504074854287661953415')

In [6]:
np.mean(get_pscores(likelihoods_A))

np.float64(1557301.2199345615)